# Trip Planner — pure LangGraph baseline

A multi-agent trip planner built on **pure LangGraph** with a local Ollama LLM (`qwen2.5:3b`). No Graxella yet — this notebook is the baseline the next one improves on.

**The graph (linear supervisor pattern):**

```
request ─→ planner ─→ flights ─→ hotels ─→ activities ─→ itinerary ─→ END
```

Each specialist node owns one tool and updates one slot of the shared `TripState`. The itinerary composer (the only LLM-driven node) weaves the three specialist outputs into a Markdown day-by-day plan.

**The realistic drift baked into `hotels`:**

The LLM was trained when `search_hotels_v1(location)` was the API. That endpoint was **sunset in Q2 2026** — every call now returns `410 Gone`. The successor is `find_accommodations(city, checkin, checkout)` (same three args, just `location` renamed to `city`), but the LLM — and the specialist node it drives — doesn't know that.

Expected outcome in this notebook: **every trip is missing hotels.** The itinerary composer will note `(hotels unavailable)` and produce a degraded plan.

---

**Prereqs.** `ollama serve` running with `qwen2.5:3b` pulled:

```bash
ollama pull qwen2.5:3b
```

## 1. Setup — make the shared `trip_planner` package importable

In [ ]:
import sys
from pathlib import Path

# The notebook lives in graxella/examples/trip_planner/. Add graxella/examples
# to sys.path so `import trip_planner.*` resolves.
_examples_root = Path.cwd().parent
if str(_examples_root) not in sys.path:
    sys.path.insert(0, str(_examples_root))

from trip_planner.tools import TOOLS, TOOLS_BY_NAME
from trip_planner.agents import (build_app, SAMPLE_QUERIES,
                                 run_batch, summarize, check_ollama, MODEL)

print(f'Model: {MODEL}')
print(f'Tools: {[t.name for t in TOOLS]}')
print(f'Sample queries: {len(SAMPLE_QUERIES)}')

In [ ]:
assert check_ollama(), 'Ollama is not reachable. Start `ollama serve` and pull qwen2.5:3b.'

## 2. Inspect the tools

Note how `search_hotels_v1`'s docstring explicitly says the API was sunset — the successor exists in `TOOLS_BY_NAME`, but the LLM's binding still points at the legacy one.

In [ ]:
for t in TOOLS:
    first_line = (t.description or '').strip().split('\n')[0]
    print(f'  {t.name:22} {first_line}')

In [ ]:
# Prove the drift directly — v1 blows up regardless of args.
try:
    TOOLS_BY_NAME['search_hotels_v1'].invoke(
        {'location': 'Paris', 'checkin': '2026-04-20', 'checkout': '2026-04-23'}
    )
except RuntimeError as e:
    print('search_hotels_v1 ->', type(e).__name__, ':', e)

## 3. Build and visualize the LangGraph app

In [ ]:
app = build_app()
print('Nodes:', list(app.get_graph().nodes))
print('Edges:')
for edge in app.get_graph().edges:
    print(f'  {edge.source} -> {edge.target}')

In [ ]:
# Render the graph as ASCII (falls back to text if Mermaid is unavailable).
try:
    print(app.get_graph().draw_ascii())
except Exception as e:
    print(f'(ASCII render unavailable: {e})')

## 4. Run one trip end-to-end

The hotels slot will come back as `(hotels unavailable: RuntimeError)` because `search_hotels_v1` raises.

In [ ]:
first = SAMPLE_QUERIES[0]
print(f'REQUEST: {first}\n')
state = app.invoke({'request': first})

print('PLAN       :', state['plan'])
print('FLIGHTS    :', state['flights'].split(chr(10))[0])
print('HOTELS     :', state['hotels'].split(chr(10))[0])
print('ACTIVITIES :', state['activities'].split(chr(10))[0])
print()
print('ERRORS     :', state.get('_errors'))
print('TIMINGS    :', state.get('_timings'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(state['itinerary']))

## 5. Run all three trips — collect metrics

Every trip should exhibit the same failure mode: `search_hotels_v1` raises, the hotels slot degrades, the itinerary composer notes the gap.

In [ ]:
results = run_batch(app)
rows = summarize(results)
rows

In [ ]:
# Pretty table (uses pandas if present, otherwise falls back to plain text).
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    for r in rows:
        print(r)

## 6. Aggregate impact of the drift

One concise number: how many trips have working hotels?

In [ ]:
n_total = len(rows)
n_hotels_ok = sum(1 for r in rows if r['hotels_ok'])
n_tool_errors = sum(r['tool_errors'] for r in rows)
total_wall = round(sum(r['wall_s'] for r in rows), 2)

print(f'Trips run             : {n_total}')
print(f'Trips with hotels ok  : {n_hotels_ok} / {n_total}')
print(f'Total tool errors     : {n_tool_errors}')
print(f'Total wall time (s)   : {total_wall}')

## 7. Show one full itinerary — hotels missing

A reader can see the concierge acknowledging the gap. This is the customer-facing symptom of a stale tool binding.

In [ ]:
worst_request, worst_state, _ = results[0]
print(f'--- {worst_request} ---')
display(Markdown(worst_state['itinerary']))

---

## Wrap-up (pure LangGraph)

- The multi-agent graph works, but the hotels specialist is bound to a **sunset API** the LLM doesn't know is broken.
- Every trip surfaces the drift as a user-visible degradation.
- To fix this in pure LangGraph the developer would need to: catch the exception in every node, hand-write a fallback ladder, retry with new arg names, or re-train / re-prompt the LLM. Every fix is *ad hoc* and lives in agent code.

**Next notebook** — `02_with_graxella.ipynb` — keeps the same tools, same graph, same LLM, and shows how one promoted rulebook entry + a one-line wrapper eliminates the entire class of failure with zero LLM retry and a full audit trail.